In [1]:
# load graph
import numpy as np
import torch

d = np.load("../data/constructs/canwell_graph.npz", allow_pickle=True)
rows, cols = d["rows"], d["cols"]
elev       = d["elev"]
slope      = d["slope"]
aspect     = d["aspect"]
doubslope  = d["doubslope"]
ddem       = d["ddem"]
is_basin   = d["is_basin"]
is_slope   = d["is_slope"]
edge_index = torch.from_numpy(d["edge_index"]).long()
H, W       = d["shape"]
N = elev.shape[0]


# node-center coordinates in meters (affine: b = d = 0 for a north-up grid)
a, b, c, dd, e, f = d["transform"]
x_coord = a*cols + c + a/2.0       # easting of each node center
y_coord = e*rows + f + e/2.0       # northing of each node center

# label set: slope nodes with a valid target (ddem)
label = is_slope & ~np.isnan(ddem)
print("N:", int(N), "| lableable slope nodes:", int(label.sum()),
      "| basin count:", int(is_basin.sum()))

N: 714421 | lableable slope nodes: 386547 | basin count: 327106


In [2]:
# random split prep
SEED = 42
VAL_FRAC = 0.25
TEST_FRAC = 0.25 # aggressive -- train pool = 0.50
rng = np.random.default_rng(SEED)

lab_idx = np.where(label)[0]         # labelable slope-node indices
perm = rng.permutation(lab_idx.size)
n_test = int(TEST_FRAC * lab_idx.size)
n_val  = int(VAL_FRAC  * lab_idx.size)
test_idx   = lab_idx[perm[:n_test]]
val_idx    = lab_idx[perm[n_test:n_test+n_val]]
train_pool = lab_idx[perm[n_test+n_val:]]      # remaining 50%

In [4]:
# learning curve knob: fraction of training pool actually used (1.0 = all)
# where "all" is the full 50%
TRAIN_FRAC = 1.0
train_idx = rng.permutation(train_pool)[:int(TRAIN_FRAC * train_pool.size)]

print(f"train {train_idx.size:,} | val {val_idx.size:,} | test {test_idx.size:,} "
      f"| train_frac {TRAIN_FRAC}")

train 193,275 | val 96,636 | test 96,636 | train_frac 1.0


In [7]:
# save labeled node splits
np.savez(f"../data/constructs/split_random_s{SEED}.npz",
         train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
         train_pool=train_pool, seed=SEED,
         val_frac=VAL_FRAC, test_frac=TEST_FRAC, train_frac=TRAIN_FRAC)
print("saved")

saved


In [9]:
# static features (fill basin terrain, scale on train stats)
asp = np.deg2rad(aspect)
asp_sin = np.sin(asp).astype(np.float32)
asp_cos = np.cos(asp).astype(np.float32)

def fill_train_mean(arr):
    m = np.nanmean(arr[train_idx])    # train = slope nodes, all real
    out = arr.astype(np.float32).copy() # don't manipulate original array
    out[np.isnan(out)] = np.float32(m)  # basin + any stray NaN -> neutral
                                     # assumes features except elev are NaN in basin
    return out

slope_f = fill_train_mean(slope)
doub_f  = fill_train_mean(doubslope)
sin_f   = fill_train_mean(asp_sin)
cos_f   = fill_train_mean(asp_cos)
elev_f  = elev.astype(np.float32)      # real everywhere, no fill

print("basin and nans filled")

basin and nans filled


In [10]:
stats = {}
def zscore(arr, name):
    m, s = float(arr[train_idx].mean()), float(arr[train_idx].std() + 1e-8)
    stats[name] = (m,s)
    return ((arr - m) / s).astype(np.float32)

elev_z  = zscore(elev_f,  "elev")
slope_z = zscore(slope_f, "slope")
doub_z  = zscore(doub_f,  "doubslope")

X_static = np.column_stack([elev_z, slope_z, sin_f, cos_f, doub_z,
                            is_basin.astype(np.float32)]).astype(np.float32)
print("X_static:", X_static.shape,
      "| basin slope_z mean ~0:", round(float(slope_z[is_basin].mean()), 3),
      "| train slope_z std ~1:", round(float(slope_z[train_idx].std()), 3))

X_static: (714421, 6) | basin slope_z mean ~0: 0.0 | train slope_z std ~1: 1.0


In [12]:
# --- target prep ----
import torch
from torch_geometric.data import Data

# target y: slope DEM, normalized on train stats (0 where no target; unused there)
y_mean = float(np.nanmean(ddem[train_idx]))
y_std  = float(np.nanstd(ddem[train_idx]) + 1e-8)
y = np.zeros(N, np.float32)
vt = ~np.isnan(ddem)
y[vt] = (ddem[vt] - y_mean) / y_std
stats["y"] = (y_mean, y_std)


In [34]:
# diff_input: basin dDEM only (as a feature) -- 0 on all slope nodes (their dDEM is target)
VISIBLE = False                           # False -> no-basin ablation (zeros everywhere)
diff = np.zeros(N, np.float32)
b_mean = b_std = None
if VISIBLE:
    bsig = is_basin & ~np.isnan(ddem)
    # b_mean = float(ddem[bsig].mean())
    b_std  = float(ddem[bsig].std() + 1e-8)
    # diff[bsig] = (ddem[bsig] - b_mean) / b_std # basin signal has own normalization
    diff[bsig] = ddem[bsig] / b_std # basin signal has own normalization

# stats["diff"] = (b_mean, b_std)
stats["diff"] = (0.0, b_std)

X = np.column_stack([X_static, diff]).astype(np.float32)   # 7 columns

In [35]:
# build data
data = Data(x = torch.from_numpy(X), edge_index=edge_index, y=torch.from_numpy(y))

def bmask(idx):
    m = torch.zeros(N, dtype=torch.bool); m[idx]=True; return m
data.train_mask = bmask(train_idx)
data.val_mask   = bmask(val_idx)
data.test_mask  = bmask(test_idx)

print("x:", tuple(data.x.shape), "| VISIBLE:", VISIBLE,
      "| y mean/std:", round(y_mean, 3), round(y_std, 3),
      "| basin diff mean/std:", None if b_mean is None else (round(b_mean, 3), round(b_std, 3)))
print("masks train/val/test:",
      int(data.train_mask.sum()), int(data.val_mask.sum()), int(data.test_mask.sum()))

x: (714421, 7) | VISIBLE: False | y mean/std: -1.957 3.742 | basin diff mean/std: None
masks train/val/test: 193275 96636 96636


In [37]:
# 5 — checkpoint prepared data + stats
import json, datetime

np.savez("../data/constructs/nobasin_stage_s42.npz",
    X_static=X_static, diff=diff, y=y,
    train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
    cols=np.array(["elev","slope","aspect_sin","aspect_cos",
                   "doubslope","is_basin","diff_input"]),
    seed=np.int64(SEED))

nobasin42_meta = {
    "created": datetime.date.today().isoformat(),
    "graph": "canwell_graph.npz",
    "split": {"seed": SEED, "train_frac": TRAIN_FRAC,
              "n_train": int(train_idx.size), "n_val": int(val_idx.size),
              "n_test": int(test_idx.size)},
    "columns": ["elev","slope","aspect_sin","aspect_cos",
                "doubslope","is_basin","diff_input"],
    #"scaling": {k: [round(v[0],6), round(v[1],6)] for k,v in stats.items()},
    "notes": {
        "diff_input": "basin dDEM, basin-normalised; 0 on slope; zeroed for no-basin run",
        "y": "slope dDEM, train-normalised; denorm = y*std + mean",
        "basin_terrain": "neutral-filled with train mean before scaling",
    },
}
with open("../data/constructs/nobasin_s42.meta.json","w") as fh:
    json.dump(nobasin42_meta, fh, indent=2)
print("checkpoint written:", X_static.shape, "| stats keys:", list(stats.keys()))

checkpoint written: (714421, 6) | stats keys: ['elev', 'slope', 'doubslope', 'y', 'diff']


In [38]:
# cell 6 -- model + loaders
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.nn import SAGEConv
from torch_geometric.loader import NeighborLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)

class CanwellSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden=64, p=0.2):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden)
        self.conv2 = SAGEConv(hidden, hidden)
        self.conv3 = SAGEConv(hidden, hidden)
        self.lin1  = Linear(hidden, hidden // 2)
        self.lin2  = Linear(hidden // 2, 1)
        self.p = p
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index)); x = F.dropout(x, self.p, self.training)
        x = F.relu(self.conv2(x, edge_index)); x = F.dropout(x, self.p, self.training)
        x = F.relu(self.conv3(x, edge_index))
        x = F.relu(self.lin1(x))
        return self.lin2(x).squeeze(1)


In [39]:
# model
model = CanwellSAGE(in_channels=data.x.size(1)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.MSELoss()

In [40]:
# loaders
train_loader = NeighborLoader(data, num_neighbors=[10,10,10], batch_size=512,
                              input_nodes=data.train_mask, shuffle=True)

val_loader   = NeighborLoader(data, num_neighbors=[10,10,10], batch_size=512,
                              input_nodes=data.val_mask, shuffle=False)

print("device:", device, "| in_channels:", data.x.size(1),
      "| params:", sum(p.numel() for p in model.parameters()),
      "| train batches:", len(train_loader))

device: cuda | in_channels: 7 | params: 19585 | train batches: 378


In [41]:
# train/validate, early stopping
def run_epoch(loader, train):
    model.train() if train else model.eval()
    total, count = 0.0, 0
    for batch in loader:
        batch = batch.to(device)
        if train: optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            out = model(batch.x, batch.edge_index)[:batch.batch_size]
            loss = criterion(out, batch.y[:batch.batch_size])
            if train:
                loss.backward(); optimizer.step()
        total += loss.item() * batch.batch_size; count += batch.batch_size
    return total/count

# set metaparameters

best_val, best_state, best_epoch, no_improve = float('inf'), None, -1, 0
patience, max_epochs = 10, 100
hist = {"train": [], "val": []}

# run
for epoch in range(max_epochs):
    tr = run_epoch(train_loader, True)
    va = run_epoch(val_loader, False)
    hist["train"].append(tr); hist["val"].append(va)
    improved = va < best_val - 1e-4
    if improved:
        best_val, best_epoch, no_improve = va, epoch, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        no_improve += 1
    if epoch % 5 == 0 or improved:
        print(f"epoch {epoch:03d} | train {tr:.4f} | val {va:.4f}" + (" *" if improved else ""))
    if no_improve >= patience:
        print(f"early stop @ epoch {epoch} | best val {best_val:.4f} @ {best_epoch}")
        break

model.load_state_dict(best_state)
                

epoch 000 | train 0.7202 | val 0.6788 *
epoch 001 | train 0.6630 | val 0.6519 *
epoch 002 | train 0.6434 | val 0.6327 *
epoch 003 | train 0.6293 | val 0.6263 *
epoch 004 | train 0.6173 | val 0.6067 *
epoch 005 | train 0.6069 | val 0.6090
epoch 006 | train 0.5965 | val 0.5878 *
epoch 007 | train 0.5879 | val 0.5781 *
epoch 008 | train 0.5777 | val 0.5748 *
epoch 009 | train 0.5723 | val 0.5582 *
epoch 010 | train 0.5651 | val 0.5593
epoch 011 | train 0.5590 | val 0.5472 *
epoch 013 | train 0.5437 | val 0.5389 *
epoch 014 | train 0.5360 | val 0.5284 *
epoch 015 | train 0.5342 | val 0.5238 *
epoch 016 | train 0.5270 | val 0.5219 *
epoch 017 | train 0.5209 | val 0.5177 *
epoch 018 | train 0.5162 | val 0.5086 *
epoch 020 | train 0.5072 | val 0.5013 *
epoch 021 | train 0.5031 | val 0.4994 *
epoch 022 | train 0.4983 | val 0.4843 *
epoch 025 | train 0.4876 | val 0.4734 *
epoch 027 | train 0.4794 | val 0.4641 *
epoch 029 | train 0.4757 | val 0.4629 *
epoch 030 | train 0.4728 | val 0.4631
epoch 

<All keys matched successfully>

In [42]:
print("best epoch:", best_epoch, "| best val:", round(best_val, 4),
      "| last val:", round(hist['val'][-1], 4),
      "| gap to best:", round(hist['val'][-1] - best_val, 4))

best epoch: 95 | best val: 0.3468 | last val: 0.3517 | gap to best: 0.005


In [44]:
# Cell 7b — continue training from current weights 04Nov26
patience, extra_epochs = 20, 200
best_val   = min(hist['val'])          # best seen so far (0.3292)
no_improve = 0
start      = len(hist['train'])        # resume numbering

for epoch in range(start, start + extra_epochs):
    tr = run_epoch(train_loader, True)
    va = run_epoch(val_loader, False)
    hist['train'].append(tr); hist['val'].append(va)
    improved = va < best_val - 1e-4
    if improved:
        best_val, best_epoch, no_improve = va, epoch, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        no_improve += 1
    if epoch % 5 == 0 or improved:
        print(f"epoch {epoch:03d} | train {tr:.4f} | val {va:.4f}" + ("  *" if improved else ""))
    if no_improve >= patience:
        print(f"early stop @ {epoch} | best val {best_val:.4f} @ {best_epoch}")
        break

model.load_state_dict(best_state)

epoch 300 | train 0.3023 | val 0.2868
epoch 305 | train 0.3034 | val 0.2912
epoch 310 | train 0.2990 | val 0.2885
epoch 315 | train 0.2996 | val 0.2907
early stop @ 319 | best val 0.2799 @ 299


<All keys matched successfully>

In [45]:
# cell 8 -- test evaluate and save
y_mean, y_std = stats["y"]
test_loader = NeighborLoader(data, num_neighbors=[-1,-1,-1], batch_size=1024,
                             input_nodes=data.test_mask, shuffle=False)

model.eval()
P, T, ID = [], [], []
with torch.no_grad():
    for batch in test_loader:
        batch=batch.to(device)
        out = model(batch.x, batch.edge_index)[:batch.batch_size]
        P.append(out.cpu()); T.append(batch.y[:batch.batch_size].cpu())
        ID.append(batch.n_id[:batch.batch_size].cpu())

pred = torch.cat(P).numpy() * y_std + y_mean
true = torch.cat(T).numpy() * y_std + y_mean
node_ids = torch.cat(ID).numpy()

mae = float(np.mean(np.abs(pred - true)))
rmse = float(np.sqrt(np.mean((pred-true) ** 2)))
r2 = float(1 - np.sum((true-pred) ** 2) / np.sum((true - true.mean()) ** 2))
print(f"{'VISIBLE' if VISIBLE else 'NO-BASIN'} test | R2 {r2:.4f} | "
      f"RMSE {rmse:.3f} m | MAE {mae:.3f} m | n={len(true):,}")

NO-BASIN test | R2 0.7198 | RMSE 1.971 m | MAE 1.353 m | n=96,636


In [46]:
tag = "visible" if VISIBLE else "nobasin"
torch.save({"state_dict": best_state, "stats": stats, "VISIBLE": VISIBLE,
            "best_epoch": best_epoch, "best_val": best_val, "hist": hist,
            "metrics": {"r2": r2, "rmse": rmse, "mae": mae}},
           f"../data/constructs/model_{tag}_s{SEED}.pt")
np.savez(f"../data/constructs/test_pred_{tag}_s{SEED}.npz",
         node_ids=node_ids, pred_m=pred, true_m=true)
print(f"saved model_{tag}_s{SEED}.pt and test_pred_{tag}_s{SEED}.npz")

saved model_nobasin_s42.pt and test_pred_nobasin_s42.npz
